## virtual proxy

- something used for lazy loading 


In [13]:
from abc import ABC, abstractmethod
import time

In [2]:
class Candidate:
    def __init__(self, name, resume):
        self.name = name
        self.resume = resume

In [9]:
c1 = Candidate("John Doe", "Resume Content for c1")
c2 = Candidate("Jane Smith", "Resume Content for c2")
c3 = Candidate("Alice Johnson", "Resume Content for c3")
c4 = Candidate("Bob Brown", "Resume Content for c4")
c5 = Candidate("Charlie Davis", "Resume Content for c5")

In [10]:
class Doc(ABC):
    @abstractmethod
    def display_resume(self):
        pass

In [29]:
class ResumeDoc(Doc):

    def __init__(self, candidate:Candidate):
        # print("Loading 50MB Resume PDF...")
        print("Loading Resume...")
        time.sleep(2)
        self.candidate = candidate

    def display_resume(self):
        print(self.candidate.resume)

In [34]:
class Client:
    def display_resume(self, document: Doc):
        document.display_resume()

In [36]:
resume_doc = ResumeDoc(c1)

client = Client()

client.display_resume(resume_doc)

Loading Resume...
Resume Content for c1


In [38]:
before_time = time.time()
client = Client()
viewers = [
    client.display_resume(ResumeDoc(c1)),
    client.display_resume(ResumeDoc(c2)),
    client.display_resume(ResumeDoc(c3)),
    client.display_resume(ResumeDoc(c4)),
    client.display_resume(ResumeDoc(c5))
]
after_time = time.time()
print(f"Time taken to load all resumes: {after_time - before_time:.2f} seconds")

Loading Resume...
Resume Content for c1
Loading Resume...
Resume Content for c2
Loading Resume...
Resume Content for c3
Loading Resume...
Resume Content for c4
Loading Resume...
Resume Content for c5
Time taken to load all resumes: 10.01 seconds


### Problem with the current approch is that we are unnessarly loading the resume 

#### Solution it should not load during the object creation it should be created withen display() function is called

In [39]:
class ResumeDocProxy(Doc):

    def __init__(self, candidate:Candidate):
        self.candidate = candidate
        self.real_resume_doc = None

    def display_resume(self):
        if self.real_resume_doc is None:
            self.real_resume_doc = ResumeDoc(self.candidate)
        self.real_resume_doc.display_resume()

In [40]:
resume = ResumeDocProxy(c1)
client = Client()
client.display_resume(resume)

Loading Resume...
Resume Content for c1


## Protection Proxy

In [41]:

from attr import dataclass


@dataclass
class Recruiter:
    name:str
    is_preminum:bool



In [42]:
# naive approch

rec1 = Recruiter("Recruiter 1", True)

In [ ]:
class RecruiterClient:
    def display_resume(self, document: Doc, recruiter: Recruiter):
        if recruiter.is_preminum:
            document.display_resume()
        else:
            print("Access Denied. Upgrade to premium to view the resume.")


In [45]:
resume_doc = ResumeDocProxy(c1)
client = RecruiterClient()
client.display_resume(resume_doc, rec1)

Loading Resume...
Resume Content for c1


### this is not a ideal approch as the client can be bypassed 

In [48]:
resume_doc.display_resume()

Resume Content for c1


### so we introduce `ResumeProtectionProxy`

In [49]:
class ResumeProtectionProxy:
    def __init__(self, resume_doc: 'ResumeDoc', recruiter: Recruiter):
        self.resume_doc = resume_doc
        self.recruiter = recruiter

    def display_resume(self):
        if self.recruiter.is_preminum:
            self.resume_doc.display_resume()
        else:
            print("Access Denied. Upgrade to premium to view the resume.")

In [51]:
recruiter_client = RecruiterClient()
resume_doc = ResumeDocProxy(c1)
security_proxy = ResumeProtectionProxy(resume_doc, rec1)

In [52]:
security_proxy.display_resume()

Loading Resume...
Resume Content for c1
